## はじめに

このノートブックでは、GlacierStyle ECサイトの各種データを加工・変換し、分析可能な形式に整形します。

**主な処理内容:**
- SNSログの感情分析・分類
- 音声ログの要約・マスキング
- 広告クリエイティブの分析
- FAQドキュメントのチャンク化
- 運用マニュアルのチャンク化
- テーブルメタデータの自動生成

In [104]:
-- ============================================================================
-- 環境設定
-- ============================================================================
-- 使用するウェアハウスとスキーマを設定
USE WAREHOUSE COMPUTE_WH;
USE SCHEMA GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA;

## 2. データの加工・変換

このセクションでは、Raw層のデータをAI機能を活用して加工・変換し、Silver/Gold層のテーブルを作成します。

### 2-1. SNSログの分類・感情分析

SNSメンション（Twitter/Instagram等）の生データに対して、以下のAI処理を実施:
- **AI_EXTRACT**: 商品名・カテゴリ・問い合わせタイプの抽出
- **AI_SENTIMENT**: 投稿の感情分析（ポジティブ/ネガティブ/ニュートラル）
- **AI_CLASSIFY**: 投稿カテゴリの分類（称賛/クレーム/質問/提案）

In [106]:
-- ============================================================================
-- SNS生ログのAI分析とGold層への保存
-- ============================================================================
-- Gold層テーブルの作成
CREATE OR REPLACE TABLE gold_sns_mentions_analyzed AS
WITH extracted_data AS (
    SELECT 
        post_id,
        platform,
        post_type,
        username,
        display_name,
        content,
        posted_at,
        likes,
        retweets,
        replies,
        hashtags,
        mentioned_products,
        media_urls,
        
        SNOWFLAKE.CORTEX.AI_EXTRACT(
            content,
            OBJECT_CONSTRUCT(
                'product_name', 'mentioned product name',
                'category', 'product category (e.g., ファッション, インテリア, テック)',
                'inquiry_type', 'inquiry type (e.g., 質問, レビュー, クレーム, 称賛, 提案)'
            )
        ) AS extracted_info
        
    FROM raw_sns_mentions
),
sentiment_analysis AS (
    SELECT 
        *,
        SNOWFLAKE.CORTEX.AI_SENTIMENT(content) AS sentiment_result
    FROM extracted_data
),
classified_data AS (
    SELECT 
        *,
        SNOWFLAKE.CORTEX.AI_CLASSIFY(
            content,
            ['称賛', 'クレーム', '質問', '提案']
        ) AS classification_result
    FROM sentiment_analysis
)
SELECT 
    post_id,
    platform,
    post_type,
    username,
    display_name,
    content,
    posted_at,
    likes,
    retweets,
    replies,
    hashtags,
    mentioned_products,
    media_urls,
    
    -- AI抽出情報
    extracted_info:response.product_name::VARCHAR AS extracted_product_name,
    extracted_info:response.category::VARCHAR AS extracted_category,
    extracted_info:response.inquiry_type::VARCHAR AS inquiry_type,
    
    -- 感情分析結果
    sentiment_result:categories[0].name::VARCHAR AS overall_sentiment,
    sentiment_result:categories[0].sentiment::VARCHAR AS sentiment,
    
    -- カテゴリ分類結果
    classification_result:labels[0]::VARCHAR AS post_category,

    -- メタデータ
    CURRENT_TIMESTAMP() AS processed_at
    
FROM classified_data;

In [107]:
-- ============================================================================
-- 分析結果の確認
-- ============================================================================
-- Gold層テーブルの内容をプレビュー
SELECT *
FROM gold_sns_mentions_analyzed
LIMIT 10;

### 2-2. 音声ログの要約・分類・マスキング

コールセンター音声ログの文字起こしデータに対して、以下のAI処理を実施:
- **AI_REDACT**: 個人情報（氏名・電話番号・住所・クレカ番号）の自動マスキング
- **AI_SENTIMENT**: 顧客感情の分析
- **AI_CLASSIFY**: 問い合わせカテゴリ分類
- **AI_AGG**: 通話内容の要約生成

In [108]:
-- ============================================================================
-- 音声ログのAI分析とGold層への保存
-- ============================================================================
CREATE OR REPLACE TABLE gold_voice_logs AS
SELECT
    -- 元のカラム（transcribed_text以外）
    * EXCLUDE transcribed_text,
    
    -- 1. AI_REDACTによる個人情報のマスキング
    -- 氏名、電話番号、住所、クレジットカード番号を自動検出・マスキング
    SNOWFLAKE.CORTEX.AI_REDACT(
        transcribed_text
    ) AS transcribed_text_masked,
    
    -- 2. AI_SENTIMENTによる顧客感情の分析
    -- マスキング済みテキストに対して感情分析を実施
    SNOWFLAKE.CORTEX.AI_SENTIMENT(
        SNOWFLAKE.CORTEX.AI_REDACT(
            transcribed_text
        )
    ) AS sentiment_result,
    
    -- 感情の詳細を抽出
    sentiment_result:categories[0].name::VARCHAR AS overall_sentiment,
    sentiment_result:categories[0].sentiment::VARCHAR AS sentiment,
    
    -- 3. AI_CLASSIFYによる問い合わせカテゴリ分類
    SNOWFLAKE.CORTEX.AI_CLASSIFY(
        transcribed_text_masked,
        ['商品に関する問い合わせ', '配送に関する問い合わせ', '返品・交換', '決済・支払い', 'アカウント・会員登録', 'クレーム', 'その他']
    ) AS classification_result,
    
    -- 分類の詳細を抽出
    classification_result:labels[0]::VARCHAR AS inquiry_category,
    
    -- 4. AI_AGGによる通話内容の要約生成（GROUP BY対象）
    AI_AGG(
        transcribed_text_masked, 
        '音声ログを400文字以内で要約してください。顧客の主な問い合わせ内容、要望、および解決状況を含めてください。'
    ) AS transcribed_text_summary,
    
    -- メタデータ
    CURRENT_TIMESTAMP() AS processed_at

FROM raw_voice_logs
GROUP BY ALL;

In [ ]:
-- ============================================================================
-- 音声ログGold層の確認
-- ============================================================================
SELECT * FROM gold_voice_logs LIMIT 10;

### 2-3. 広告メタデータの抽出

広告クリエイティブデータに対して、以下のAI処理を実施:
- **AI_COMPLETE**: 画像からビジュアル要素を抽出（色・構図・人物・印象等）
- **AI_CLASSIFY**: 画像スタイルの分類
- **AI_EXTRACT**: コピーテキストから訴求ポイント・CTA・キーワード等を抽出
- **AI_SENTIMENT**: コピーの感情分析

In [83]:
CREATE OR REPLACE TEMP TABLE silver_ad_visual_analysis AS
WITH image_files AS (
    SELECT 
        relative_path,
        SPLIT_PART(relative_path, '/', -1) AS file_name,
        file_url,
        size,
        last_modified
    FROM DIRECTORY(@DATA_STAGE)
    WHERE relative_path ILIKE 'ad_images/%.png'
       OR relative_path ILIKE 'ad_images/%.jpg'
       OR relative_path ILIKE 'ad_images/%.jpeg'
)
SELECT 
    file_name,
    relative_path,
    file_url,
    size,
    last_modified,
    
    -- AI_COMPLETEで画像からビジュアル要素を抽出
    AI_COMPLETE(
        'openai-gpt-4.1',
        '以下の広告画像を分析し、JSON形式で以下の要素を抽出してください：
        {
            "メインカラー": "画像の主要な色（最大3色）",
            "構図タイプ": "中央配置/三分割/対角線/その他",
            "人物有無": "あり/なし",
            "人物の特徴": "性別、年齢層、表情など（人物がいる場合）",
            "商品配置": "商品の位置と見せ方",
            "背景スタイル": "単色/グラデーション/写真/イラストなど",
            "全体的な印象": "高級感/カジュアル/モダン/ナチュラルなど",
            "ターゲット層推定": "想定されるターゲット層"
        }
        JSON形式のみで回答してください。',
        TO_FILE('@DATA_STAGE', relative_path)
        
    ) AS visual_analysis_raw,
    TRY_PARSE_JSON(visual_analysis_raw) visual_analysis_raw_json,
    
    -- 画像分類（スタイル別）
    AI_CLASSIFY(
        TO_FILE('@DATA_STAGE', relative_path),
        ['商品フォーカス', 'ライフスタイル', 'テキスト中心', 'キャンペーン訴求', 'ブランドイメージ']
    ) AS image_style_classification,
    
    -- 分類結果を抽出
    image_style_classification:labels[0]::VARCHAR AS image_style,
    
    CURRENT_TIMESTAMP() AS analyzed_at

FROM image_files;


In [84]:
CREATE OR REPLACE TEMP TABLE silver_ad_copy_analysis AS
SELECT 
    creative_id,
    creative_name,
    copy_text,
    headline,
    cta_text,
    
    -- AI_EXTRACTでコピー要素を抽出
    SNOWFLAKE.CORTEX.AI_EXTRACT(
        copy_text || ' ' || headline || ' ' || cta_text,
        OBJECT_CONSTRUCT(
            'appeal_type', 'main appeal point (e.g., 品質訴求, 価格訴求, 限定訴求, 感情訴求)',
            'cta_type', 'call-to-action type (e.g., 購入促進, 情報収集, 会員登録)',
            'keywords', 'main keywords (comma separated)',
            'target_emotion', 'target emotion (e.g., 安心感, ワクワク感, 緊急感)',
            'usp', 'unique selling proposition'
        )
    ) AS copy_extraction,
    
    -- 抽出結果を個別カラムに展開
    copy_extraction:response.appeal_type::VARCHAR AS appeal_type,
    copy_extraction:response.cta_type::VARCHAR AS cta_type,
    copy_extraction:response.keywords::VARCHAR AS keywords,
    copy_extraction:response.target_emotion::VARCHAR AS target_emotion,
    copy_extraction:response.usp::VARCHAR AS usp,
    
    -- コピーの感情分析
    SNOWFLAKE.CORTEX.AI_SENTIMENT(copy_text) AS copy_sentiment,
    copy_sentiment:categories[0].name::VARCHAR AS sentiment,
    copy_sentiment:categories[0].sentiment::VARCHAR AS sentiment_score,
    
    -- コピーのスタイル分類
    SNOWFLAKE.CORTEX.AI_CLASSIFY(
        copy_text,
        ['直接的訴求', '感情的訴求', '論理的訴求', '限定・緊急訴求', 'ストーリー訴求']
    ) AS copy_style_result,
    copy_style_result:labels::VARCHAR AS copy_style,
    
    CURRENT_TIMESTAMP() AS analyzed_at

FROM raw_ad_creatives;


In [89]:
CREATE OR REPLACE TABLE gold_ad_creative_analysis AS
SELECT 
    -- 広告クリエイティブ基本情報
    c.creative_id,
    c.creative_name,
    c.creative_type,
    c.campaign_id,
    c.platform,
    c.target_segment,
    
    -- テキスト関連
    c.copy_text,
    c.headline,
    c.cta_text,
    
    -- コピー分析結果
    t.appeal_type,
    t.cta_type,
    t.keywords,
    t.target_emotion,
    t.usp,
    t.copy_style,
    t.sentiment,
    
    -- パフォーマンスデータ
    c.impressions,
    c.clicks,
    c.conversions,
    c.spend,
    CASE WHEN c.impressions > 0 THEN (c.clicks::FLOAT / c.impressions) * 100 ELSE 0 END AS ctr,
    CASE WHEN c.clicks > 0 THEN (c.conversions::FLOAT / c.clicks) * 100 ELSE 0 END AS cvr,
    CASE WHEN c.conversions > 0 THEN c.spend / c.conversions ELSE 0 END AS cpa,

    -- 画像データ
    v.visual_analysis_raw_json,
    v.image_style
    
    -- メタデータ
    CURRENT_TIMESTAMP() AS processed_at

FROM raw_ad_creatives c
LEFT JOIN silver_ad_copy_analysis t ON c.creative_id = t.creative_id
LEFT JOIN silver_ad_visual_analysis v ON SPLIT_PART(c.image_file_path, '/', -1) = SPLIT_PART(v.relative_path, '/', -1) 

### 2-4. FAQドキュメントのチャンク化

FAQドキュメントをカテゴリ別に集約し、AI_AGGで要約を生成:
- カテゴリ別の質問項目サマリーをJSON形式で出力
- RAG検索用のコンテキスト情報として活用

In [93]:
-- ============================================================================
-- FAQドキュメントデータの確認
-- ============================================================================
SELECT *
FROM raw_faq_documents_parsed
LIMIT 100;

In [99]:
-- ============================================================================
-- FAQドキュメントのGold層テーブル作成
-- ============================================================================
-- カテゴリ別にAI_AGGで質問項目を要約し、元データと結合
CREATE OR REPLACE TABLE gold_faq_documents AS
WITH category_summary AS (
    SELECT
        category,
        AI_AGG(
            content_chunk,
            'どういった質問項目があるのかを中心に要約してください。出力形式はJSON形式にしてください'
        ) AS summary_category
    FROM raw_faq_documents_parsed
    GROUP BY category
)
SELECT
    t1.*,
    t2.summary_category
FROM raw_faq_documents_parsed AS t1
INNER JOIN category_summary AS t2
    ON t1.category = t2.category;

### 2-5. 運用マニュアルのチャンク化

運用マニュアルを部門別に集約し、AI_AGGで章・節単位の要約を生成:
- 部門別のマニュアル内容サマリーをJSON形式で出力
- 社内ナレッジ検索用のコンテキスト情報として活用

In [102]:
-- ============================================================================
-- 運用マニュアルのGold層テーブル作成
-- ============================================================================
-- 部門別にAI_AGGで章・節を要約し、元データと結合
CREATE OR REPLACE TABLE gold_operation_manuals AS
WITH department_summary AS (
    SELECT
        department,
        AI_AGG(
            content_chunk,
            '各種章や節あたりで要約してください。出力形式はJSON形式にしてください'
        ) AS summary_department
    FROM raw_operation_manuals_parsed
    GROUP BY department
)
SELECT
    t1.*,
    t2.summary_department
FROM raw_operation_manuals_parsed AS t1
INNER JOIN department_summary AS t2
    ON t1.department = t2.department;

In [103]:
-- ============================================================================
-- 運用マニュアルGold層の確認
-- ============================================================================
SELECT *
FROM gold_operation_manuals
LIMIT 10;

### 2-6. 商品マスタとの突合（名寄せ）

SNSメンションから抽出した商品名と商品マスタをAI_SIMILARITYで突合:
- カテゴリでフィルタ後、商品名の類似度を計算
- 類似度が高い順にソート

In [122]:
-- ============================================================================
-- SNSメンションと商品マスタの突合（名寄せ）
-- ============================================================================
-- AI_SIMILARITYを使用して商品名の類似度を計算
CREATE OR REPLACE TABLE gold_sns_mentions_with_product_master AS
WITH product_match AS (
    SELECT 
        *
    FROM gold_sns_mentions_analyzed t1
    INNER JOIN dim_products t2
        ON t2.category_l1 = t1.extracted_category
)
SELECT
    *,
    AI_SIMILARITY(extracted_product_name, product_name) AS similarity
FROM product_match
ORDER BY post_id, similarity DESC;

In [124]:
-- ============================================================================
-- 商品マスタ突合結果の確認
-- ============================================================================
SELECT * FROM gold_sns_mentions_with_product_master ORDER BY post_id LIMIT 1000;

## まとめ

このノートブックでは、GlacierStyle ECサイトのRaw層データをSnowflake Cortex AI機能を活用して加工・変換し、分析可能なSilver/Gold層テーブルを作成しました。

### 作成したテーブル一覧

**Silver層（一時テーブル）**
- `silver_ad_visual_analysis`: 広告画像のビジュアル分析結果
- `silver_ad_copy_analysis`: 広告コピーのテキスト分析結果

**Gold層**
- `gold_sns_mentions_analyzed`: SNSメンションのAI分析結果（感情・分類・抽出情報）
- `gold_voice_logs`: 音声ログの要約・マスキング・分類結果
- `gold_ad_creative_analysis`: 広告クリエイティブの統合分析結果
- `gold_faq_documents`: FAQドキュメントのカテゴリ別要約
- `gold_operation_manuals`: 運用マニュアルの部門別要約
- `gold_sns_mentions_with_product_master`: SNSメンションと商品マスタの突合結果

### 使用したCortex AI関数

- `AI_EXTRACT`: テキストから構造化情報を抽出
- `AI_SENTIMENT`: 感情分析（ポジティブ/ネガティブ/ニュートラル）
- `AI_CLASSIFY`: テキスト・画像のカテゴリ分類
- `AI_REDACT`: 個人情報の自動マスキング
- `AI_AGG`: テキストの要約生成
- `AI_COMPLETE`: 画像分析・自由形式のAI処理
- `AI_SIMILARITY`: テキスト間の類似度計算
- `AI_GENERATE_TABLE_DESC`: テーブル・カラム説明の自動生成
- `TRANSLATE`: 多言語翻訳

### 次のステップ

- **Part 3**: 各種テーブルへのメタデータ自動付与